In [1]:
import pandas as pd
import numpy as np
import os,glob,nrrd
from tqdm import tqdm

In [2]:
anno,header = nrrd.read("..\\CLA\\annotation_10.nrrd")
dfarea = pd.read_excel("..\\CLA\\brain_areas.xlsx")

In [3]:
neurons = glob.glob(r"J:\BLA_four_types\BLA_swc_mirrow\*")

In [5]:
# 转 neuron为不同edge  CA1 限定
# neurons = glob.glob(r"H:\20230302_figs\20230303_POA_VMH_combine\swc\*")
dfcsvpath = r"J:\BLA_four_types\all_csv2"
dfcombine = pd.read_excel(r"D:\python\pre-cube length\level_8_map_cortex friendly_change_swc.xlsx",index_col=0)
subarea = dfcombine.child.to_list()
def get_branth(x):
    global bratch_ID
    tmp = x.parent - x.ID
    if tmp < -2:
        bratch_ID += 1
    return bratch_ID

def get_side(x):
    if x.ML <= 5700:
        return "left"
    else: 
        return "right"

def get_area_ID(x):
    try:
        area_ID = anno[round(x.AP/10),round(x.DV/10),round(x.ML/10)]
        return area_ID
    except:
        return 0

def get_area_name(x):
    if x.area_ID !=0:
        area_name = dfarea.loc[dfarea.ID == x.area_ID].region.values[0]   
        return area_name
    else:
        return "unname"

def get_name_use(x):
    if x.area_name !="unname":
        if x.area_name in subarea:
            name_use = dfcombine.loc[dfcombine.child == x.area_name,"father"].tolist()[0]  
            return name_use
        else:
            return x.area_name
    else:
        return "unname"


for neuron in tqdm(neurons):
    # print(neuron)
    file = neuron.split("\\")[-1].split(".")[0]
    dfneuron = pd.read_csv(neuron,sep = " ",header= None)
    dfneuron.columns = ["ID","type","AP","DV","ML","R","parent"]
    dfneuron.loc[:, "area_ID"] = dfneuron.apply(get_area_ID, axis=1)
    dfneuron.loc[:, "area_name"] = dfneuron.apply(get_area_name, axis=1)
    dfneuron.loc[:, "name_use"] = dfneuron.apply(get_name_use, axis=1)
    # dfneuron.loc[:, "in_not"] = dfneuron.apply(get_in_or_not, axis=1)
    soma = dfneuron.loc[dfneuron.parent == -1].name_use.values[0]
    bratch_ID = 1
    dfneuron.loc[:, "bratch_ID"] = dfneuron.apply(get_branth, axis=1)
    # dfneuron.loc[:, "area_ID"] = dfneuron.apply(get_area_ID, axis=1)
    # dfneuron.loc[:, "area_name"] = dfneuron.apply(get_area_name, axis=1)
    
    dfneuron.loc[:, "side"] = dfneuron.apply(get_side, axis=1)
    dfneuron.to_csv(os.path.join(dfcsvpath,file+".csv"))

100%|██████████| 92/92 [06:31<00:00,  4.25s/it]


In [ ]:
# 转 neuron为不同edge
# neurons = glob.glob(r"H:\20230302_figs\20230303_POA_VMH_combine\swc\*")
dfcsvpath = r"J:\260228_Hip\all_csv"
dfcombine = pd.read_excel(r"D:\python\pre-cube length\level_8_map_cortex friendly_change_swc.xlsx",index_col=0)
subarea = dfcombine.child.to_list()
def get_branth(x):
    global bratch_ID
    tmp = x.parent - x.ID
    if tmp < -2:
        bratch_ID += 1
    return bratch_ID

def get_side(x):
    if x.ML <= 5700:
        return "left"
    else: 
        return "right"

def get_area_ID(x):
    try:
        area_ID = anno[round(x.AP/10),round(x.DV/10),round(x.ML/10)]
        return area_ID
    except:
        return 0

def get_area_name(x):
    if x.area_ID !=0:
        area_name = dfarea.loc[dfarea.ID == x.area_ID].region.values[0]   
        return area_name
    else:
        return "unname"

def get_name_use(x):
    if x.area_name !="unname":
        if x.area_name in subarea:
            name_use = dfcombine.loc[dfcombine.child == x.area_name,"father"].tolist()[0]  
            return name_use
        else:
            return x.area_name
    else:
        return "unname"


for neuron in tqdm(neurons):
    # print(neuron)
    file = neuron.split("\\")[-1].split(".")[0]
    dfneuron = pd.read_csv(neuron,sep = " ",header= None)
    dfneuron.columns = ["ID","type","AP","DV","ML","R","parent"]
    # dfneuron.loc[:, "in_not"] = dfneuron.apply(get_in_or_not, axis=1)
    bratch_ID = 1
    dfneuron.loc[:, "bratch_ID"] = dfneuron.apply(get_branth, axis=1)
    dfneuron.loc[:, "area_ID"] = dfneuron.apply(get_area_ID, axis=1)
    dfneuron.loc[:, "area_name"] = dfneuron.apply(get_area_name, axis=1)
    dfneuron.loc[:, "name_use"] = dfneuron.apply(get_name_use, axis=1)
    dfneuron.loc[:, "side"] = dfneuron.apply(get_side, axis=1)
    dfneuron.to_csv(os.path.join(dfcsvpath,file+".csv"))

100%|██████████| 192/192 [47:33<00:00, 14.86s/it] 


In [6]:
%reset

In [6]:
files = glob.glob(r"J:\BLA_four_types\all_csv2\*")
outpath = r"J:\BLA_four_types\csv_new"
os.makedirs(outpath,exist_ok=True)

for filetmp in tqdm(files):
    dftmp = pd.read_csv(filetmp,index_col=0)
    filename = filetmp.split("\\")[-1]
    inlist = dftmp.parent.unique()
    IDlist = dftmp.ID.tolist()
    dftmp["terminal"] = [0 if item in inlist else 1 for item in IDlist]
    dftmp.to_csv(os.path.join(outpath,filename))
    

100%|██████████| 92/92 [00:27<00:00,  3.39it/s]


In [ ]:
## get terminal infor
files = glob.glob(r"J:\BLA_Cck\csv_new\*")
for file in files:
    

In [13]:
dftmp.type.value_counts()

2    57535
3     3410
1        1
Name: type, dtype: int64

In [14]:
dftmp.loc[dftmp.terminal == 1].type.value_counts()

2    337
3     62
Name: type, dtype: int64

In [17]:
# dftmp = dftmp.drop("Unnamed: 0.1",axis=1)
dftmpnew = dftmp.loc[dftmp.terminal == 1]
dftmpnew["soma_AP"] = dftmp.iloc[0,:].AP
dftmpnew["soma_DV"] = dftmp.iloc[0,:].DV
dftmpnew["soma_ML"] = dftmp.iloc[0,:].ML
filename = filetmp.split(".")[0]
dftmpnew["filename"] = filename


C:\Users\zljia\AppData\Local\Temp\ipykernel_281688\768861316.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dftmpnew["soma_AP"] = dftmp.iloc[0,:].AP
C:\Users\zljia\AppData\Local\Temp\ipykernel_281688\768861316.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dftmpnew["soma_DV"] = dftmp.iloc[0,:].DV
C:\Users\zljia\AppData\Local\Temp\ipykernel_281688\768861316.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = val

In [18]:
dftmpnew

,ID,type,AP,DV,ML,R,parent,bratch_ID,area_ID,area_name,name_use,side,terminal,soma_AP,soma_DV,soma_ML,filename
313,314,3,7476.36,6618.32,2189.66,0.378906,313,11,451,BLAv,BLA,left,1,7412.1,6568.3,2227.68,J:\BLA_Cck\csv\252578_033
464,465,3,7280.50,6569.82,2217.38,0.324219,464,19,451,BLAv,BLA,left,1,7412.1,6568.3,2227.68,J:\BLA_Cck\csv\252578_033
528,529,3,7534.78,6624.58,2271.40,0.007812,528,26,451,BLAv,BLA,left,1,7412.1,6568.3,2227.68,J:\BLA_Cck\csv\252578_033
582,583,3,7460.62,6499.12,2248.60,0.406250,582,28,451,BLAv,BLA,left,1,7412.1,6568.3,2227.68,J:\BLA_Cck\csv\252578_033
673,674,3,7474.12,6532.08,2112.84,0.296875,673,31,451,BLAv,BLA,left,1,7412.1,6568.3,2227.68,J:\BLA_Cck\csv\252578_033
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60390,60391,2,5798.02,6244.84,7568.92,0.296875,60390,783,298,MA,MA,right,1,7412.1,6568.3,2227.68,J:\BLA_Cck\csv\252578_033
60670,60671,2,6595.60,5804.48,8736.42,0.648438,60670,785,303,BLAa,BLA,right,1,7412.1,6568.3,2227.68,J:\BLA_Cck\csv\252578_033
60933,60934,2,6613.84,6071.88,8822.94,0.406250,60933,786,303,BLAa,BLA,right,1,7412.1,6568.3,2227.68,J:\BLA_Cck\csv\252578_033
60936,60937,2,5671.34,6270.72,7533.28,0.570312,60936,787,298,MA,MA,right,1,7412.1,6568.3,2227.68,J:\BLA_Cck\csv\252578_033


In [7]:
## combine csv files
os.chdir(r"J:\BLA_four_types\csv_new")
files = os.listdir()


In [25]:
dftmpnew

,ID,type,AP,DV,ML,R,parent,bratch_ID,area_ID,area_name,name_use,side,terminal,soma_AP,soma_DV,soma_ML,filename
313,314,3,7476.36,6618.32,2189.66,0.378906,313,11,451,BLAv,BLA,left,1,7412.1,6568.3,2227.68,252578_033
464,465,3,7280.50,6569.82,2217.38,0.324219,464,19,451,BLAv,BLA,left,1,7412.1,6568.3,2227.68,252578_033
528,529,3,7534.78,6624.58,2271.40,0.007812,528,26,451,BLAv,BLA,left,1,7412.1,6568.3,2227.68,252578_033
582,583,3,7460.62,6499.12,2248.60,0.406250,582,28,451,BLAv,BLA,left,1,7412.1,6568.3,2227.68,252578_033
673,674,3,7474.12,6532.08,2112.84,0.296875,673,31,451,BLAv,BLA,left,1,7412.1,6568.3,2227.68,252578_033
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60390,60391,2,5798.02,6244.84,7568.92,0.296875,60390,783,298,MA,MA,right,1,7412.1,6568.3,2227.68,252578_033
60670,60671,2,6595.60,5804.48,8736.42,0.648438,60670,785,303,BLAa,BLA,right,1,7412.1,6568.3,2227.68,252578_033
60933,60934,2,6613.84,6071.88,8822.94,0.406250,60933,786,303,BLAa,BLA,right,1,7412.1,6568.3,2227.68,252578_033
60936,60937,2,5671.34,6270.72,7533.28,0.570312,60936,787,298,MA,MA,right,1,7412.1,6568.3,2227.68,252578_033


In [8]:
dfall = pd.DataFrame()
for filetmp in files:
    dftmp = pd.read_csv(filetmp,index_col=0)
    # dftmp = dftmp.drop("Unnamed: 0.1",axis=1)
    dftmpnew = dftmp.loc[dftmp.terminal == 1]
    dftmpnew["soma_AP"] = dftmp.iloc[0,:].AP
    dftmpnew["soma_DV"] = dftmp.iloc[0,:].DV
    dftmpnew["soma_ML"] = dftmp.iloc[0,:].ML
    filename = filetmp.split(".")[0]
    dftmpnew["filename"] = filename
    # dftmpnew["geno"] = filename.split("_")[2]
    # dftmpnew["geno"] = filename.split("_")[2]
    dfall = pd.concat([dfall,dftmpnew],axis=0)


C:\Users\zljia\AppData\Local\Temp\ipykernel_21532\1152209363.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dftmpnew["soma_AP"] = dftmp.iloc[0,:].AP
C:\Users\zljia\AppData\Local\Temp\ipykernel_21532\1152209363.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dftmpnew["soma_DV"] = dftmp.iloc[0,:].DV
C:\Users\zljia\AppData\Local\Temp\ipykernel_21532\1152209363.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = val

In [9]:

dfall.to_csv("..//all_terminal_infor_combine.csv")

In [41]:
dfall["name_side"] = dfall["name_use"]+"_"+dfall["side"]

In [42]:
dfall

,ID,type,AP,DV,ML,R,parent,bratch_ID,area_ID,area_name,name_use,side,terminal,soma_AP,soma_DV,soma_ML,filename,name_side
293,294,3,6682.82,6054.16,2563.52,0.648438,293,8,303,BLAa,BLA,left,1,6845.74,6292.12,2574.60,251039_051,BLA_left
345,346,3,6726.18,6288.20,2571.92,0.433594,345,10,303,BLAa,BLA,left,1,6845.74,6292.12,2574.60,251039_051,BLA_left
458,459,3,6853.02,6153.94,2581.64,1.136719,458,13,303,BLAa,BLA,left,1,6845.74,6292.12,2574.60,251039_051,BLA_left
533,534,3,6754.12,6361.72,2651.54,0.324219,533,15,703,CTXsp,CTXsp,left,1,6845.74,6292.12,2574.60,251039_051,CTXsp_left
1564,1565,2,6908.06,6454.82,2923.98,0.007812,1564,17,327,BMAa,BMA,left,1,6845.74,6292.12,2574.60,251039_051,BMA_left
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60390,60391,2,5798.02,6244.84,7568.92,0.296875,60390,783,298,MA,MA,right,1,7412.10,6568.30,2227.68,252578_033,MA_right
60670,60671,2,6595.60,5804.48,8736.42,0.648438,60670,785,303,BLAa,BLA,right,1,7412.10,6568.30,2227.68,252578_033,BLA_right
60933,60934,2,6613.84,6071.88,8822.94,0.406250,60933,786,303,BLAa,BLA,right,1,7412.10,6568.30,2227.68,252578_033,BLA_right
60936,60937,2,5671.34,6270.72,7533.28,0.570312,60936,787,298,MA,MA,right,1,7412.10,6568.30,2227.68,252578_033,MA_right


In [44]:
dfcombine_new = dfall.pivot_table(columns="name_side",index = "filename",values="terminal",aggfunc="count").fillna(0)

In [46]:
dfcombine_new.to_excel("terminal_infor_table.xlsx")

In [38]:
dfall.pivot_table(columns="area_name",index = "filename",values="terminal",aggfunc="count").fillna(0)

area_name,AAA,ACAd1,ACAd2/3,ACAd5,ACAd6a,ACAv1,ACAv2/3,ACAv5,ACAv6a,ACAv6b,...,och,opt,or,root,rust,scwm,st,stc,unname,vtd
filename,,,,,,,,,,,,,,,,,,,,,
251038_001,0.0,2.0,2.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
251038_002,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
251038_004,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
251038_005,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
251038_006,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
252578_026,1.0,0.0,0.0,0.0,0.0,4.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
252578_027,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
252578_028,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [41]:
dftmpnew["filename"] = filename
dftmpnew["geno"] = filename.split("_")[2]


,ID,type,AP,DV,ML,R,parent,bratch_ID,area_ID,area_name,name_use,side,terminal,soma_AP,soma_DV,soma_ML
832,833,2,5091.883789,5928.304199,5078.993652,1.078125,832,8,1109,PS,PS,left,1,5198.300781,6167.281738,5326.173828
839,840,2,5101.466797,5480.867676,5435.292969,1.285156,839,9,72,ADP,ADP,left,1,5198.300781,6167.281738,5326.173828
1859,1860,2,5089.581055,5917.661621,5085.225586,0.816406,1859,15,1109,PS,PS,left,1,5198.300781,6167.281738,5326.173828
1907,1908,2,4953.658203,5128.794434,5599.269043,0.933594,1907,17,564,MS,MS,left,1,5198.300781,6167.281738,5326.173828
2072,2073,2,5045.105957,6009.875977,5272.760742,0.847656,2072,23,523,MPO,MPO,left,1,5198.300781,6167.281738,5326.173828
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51637,51638,2,7329.959473,6263.396973,6351.452148,1.285156,51637,1309,946,PH,PH,right,1,5198.300781,6167.281738,5326.173828
51647,51648,2,7321.141602,6268.345215,6366.808594,0.992188,51647,1311,946,PH,PH,right,1,5198.300781,6167.281738,5326.173828
51649,51650,2,7326.241211,6272.154297,6361.550293,0.496094,51649,1312,946,PH,PH,right,1,5198.300781,6167.281738,5326.173828
51659,51660,2,7324.324219,6274.187500,6361.359375,1.398438,51659,1313,946,PH,PH,right,1,5198.300781,6167.281738,5326.173828


In [38]:
dftmp.iloc[0,:].AP

5198.300781

C:\Users\win7\AppData\Local\Programs\Python\Python37\Lib\site-packages\ipykernel_launcher.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  """Entry point for launching an IPython kernel.
C:\Users\win7\AppData\Local\Programs\Python\Python37\Lib\site-packages\ipykernel_launcher.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  
C:\Users\win7\AppData\Local\Programs\Python\Python37\Lib\site-packages\ipykernel_launcher.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a Da

In [23]:
for filetmp in  files:
    

['180712_001_Esr1_MPN.csv',
 '180712_002_Esr1_MPN.csv',
 '180712_004_Esr1_MPN.csv',
 '180712_005_Esr1_MPN.csv',
 '180712_006_Esr1_VMPO.csv',
 '180712_007_Esr1_MPN.csv',
 '180712_008_Esr1_MPN.csv',
 '180712_021_Esr1_MPO.csv',
 '180712_022_Esr1_MPN.csv',
 '180712_023_Esr1_MPN.csv',
 '180712_024_Esr1_MPN.csv',
 '180712_025_Esr1_MPN.csv',
 '180712_026_Esr1_BST.csv',
 '180712_027_Esr1_MPN.csv',
 '180712_028_Esr1_MPN.csv',
 '180712_029_Esr1_MPN.csv',
 '180712_031_Esr1_MPN.csv',
 '180712_032_Esr1_MPN.csv',
 '180712_036_Esr1_MPN.csv',
 '180712_037_Esr1_MPN.csv',
 '180712_038_Esr1_MPN.csv',
 '180712_039_Esr1_MPO.csv',
 '180712_040_Esr1_MPO.csv',
 '180712_041_Esr1_MPN.csv',
 '180712_042_Esr1_MPN.csv',
 '180712_043_Esr1_MPN.csv',
 '180712_044_Esr1_BST.csv',
 '180712_045_Esr1_BST.csv',
 '180712_048_Esr1_HY.csv',
 '180712_050_Esr1_BST.csv',
 '180712_051_Esr1_MPN.csv',
 '180712_052_Esr1_PD.csv',
 '180712_053_Esr1_MPO.csv',
 '180712_054_Esr1_MPN.csv',
 '180712_055_Esr1_HY.csv',
 '180712_056_Esr1_MPN.

In [11]:
if dftmp.parent.unique()

array([   -1,     1,     2, ..., 51666, 51667, 51668], dtype=int64)

In [19]:
dftmp["terminal"] = [0 if item in dftmp.parent.unique() else 1 for item in dftmp.ID.tolist()]

In [18]:
inlist = dftmp.parent.unique()
IDlist = dftmp.ID.tolist()
dftmp["terminal"] = [0 if item in inlist else 1 for item in IDlist]

In [16]:
dftmp

,ID,type,AP,DV,ML,R,parent,bratch_ID,area_ID,area_name,name_use,side,terminal
0,1,1,5198.300781,6167.281738,5326.173828,3.414063,-1,1,515,MPN,MPN,left,0
1,2,2,5196.447266,6167.419434,5326.122559,2.042969,1,1,515,MPN,MPN,left,0
2,3,2,5194.444336,6166.463867,5325.943359,0.933594,2,1,523,MPO,MPO,left,0
3,4,2,5193.116211,6164.957520,5325.913574,1.136719,3,1,523,MPO,MPO,left,0
4,5,2,5191.160645,6163.651855,5326.088867,0.496094,4,1,523,MPO,MPO,left,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
51664,51665,2,7328.383301,6280.264648,6361.663086,0.960938,51664,1314,946,PH,PH,right,0
51665,51666,2,7329.422363,6280.909180,6360.434082,0.320313,51665,1314,946,PH,PH,right,0
51666,51667,2,7329.641602,6281.489746,6359.133789,0.757813,51666,1314,946,PH,PH,right,0
51667,51668,2,7330.269043,6283.376465,6358.964844,0.816406,51667,1314,946,PH,PH,right,0


In [12]:
dftmp

,ID,type,AP,DV,ML,R,parent,bratch_ID,area_ID,area_name,name_use,side
0,1,1,5198.300781,6167.281738,5326.173828,3.414063,-1,1,515,MPN,MPN,left
1,2,2,5196.447266,6167.419434,5326.122559,2.042969,1,1,515,MPN,MPN,left
2,3,2,5194.444336,6166.463867,5325.943359,0.933594,2,1,523,MPO,MPO,left
3,4,2,5193.116211,6164.957520,5325.913574,1.136719,3,1,523,MPO,MPO,left
4,5,2,5191.160645,6163.651855,5326.088867,0.496094,4,1,523,MPO,MPO,left
...,...,...,...,...,...,...,...,...,...,...,...,...
51664,51665,2,7328.383301,6280.264648,6361.663086,0.960938,51664,1314,946,PH,PH,right
51665,51666,2,7329.422363,6280.909180,6360.434082,0.320313,51665,1314,946,PH,PH,right
51666,51667,2,7329.641602,6281.489746,6359.133789,0.757813,51666,1314,946,PH,PH,right
51667,51668,2,7330.269043,6283.376465,6358.964844,0.816406,51667,1314,946,PH,PH,right
